# BLS Pipeline: 100 NASA Confirmed Planets

**Purpose**: Select 100 NASA confirmed transiting planets, download their TESS light curves, run the same BLS pipeline as Person2, and output a CSV with columns **identical to `master_candidates.csv`**.

**Key difference from `批量BLS_500目标.ipynb`**: Step 2 is replaced — instead of random TIC stars, targets come from NASA Exoplanet Archive (`pscomppars` = confirmed planets table).

**Workflow**:
```
1. Query pscomppars -> filter: Transit method + period 0.5-40d -> de-duplicate by host -> sample 100
2. For each host star:
   a. Search MAST for SPOC light curves by hostname
   b. Download + flatten + run BLS + bootstrap errors + vet (False Positive flag)
   c. Append 1-row summary to the batch table
3. Save -> outputs/confirmed_100.csv  (same 8 columns as Person1's master_candidates.csv)
```

**Output**: `outputs/confirmed_100.csv`

## 0. Install Dependencies

In [8]:
%pip install -q numpy pandas astropy astroquery lightkurve

Note: you may need to restart the kernel to use updated packages.


## 1. Configuration

In [9]:
import os, time as timer, warnings
import numpy as np
import pandas as pd
from astropy.timeseries import BoxLeastSquares
import lightkurve as lk

warnings.filterwarnings('ignore')

# ---------------- Parameters ----------------
N_TARGETS          = 50          # number of confirmed planets to process
RANDOM_SEED         = 99           # reproducibility
N_BOOTSTRAP         = 50           # bootstrap draws per target (50 = fast screening; 200 = final)
MIN_PERIOD          = 0.5          # days
MAX_PERIOD          = 40.0         # days (v2 widened from 15)
SNR_THRESHOLD       = 7.0          # depth_snr below this -> False Positive
PER_TARGET_TIMEOUT  = 180          # seconds hard timeout per target
BATCH_SIZE          = 20           # save intermediate CSV every N targets
TESS_CACHE_DIR      = 'tess_cache'
OUTPUT_DIR          = 'outputs'
os.makedirs(TESS_CACHE_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
OUTPUT_CSV = os.path.join(OUTPUT_DIR, 'output.csv')
CHECKPOINT_CSV = os.path.join(OUTPUT_DIR, 'checkpoint.csv')

print(f'Will process {N_TARGETS} NASA confirmed planets')
print(f'BLS period range: {MIN_PERIOD} - {MAX_PERIOD} days')
print(f'Timeout per target: {PER_TARGET_TIMEOUT}s, batch size: {BATCH_SIZE}')

Will process 50 NASA confirmed planets
BLS period range: 0.5 - 40.0 days
Timeout per target: 180s, batch size: 20


## 2. Get 100 NASA Confirmed Planets (replaces random TIC query)

Query `pscomppars` (the confirmed planets table), filter for transit method + period in TESS-detectable range, de-duplicate by host star, randomly sample 100.

In [10]:
from astroquery.ipac.nexsci.nasa_exoplanet_archive import NasaExoplanetArchive

def _safe_val(x):
    '''Safely extract float from astropy Quantity / masked / None value.'''
    if x is None:
        return np.nan
    if np.ma.is_masked(x):
        return np.nan
    if hasattr(x, 'value'):       # astropy Quantity (e.g. pl_orbper has unit 'd')
        return float(x.value)
    try:
        return float(x)
    except (TypeError, ValueError):
        return np.nan

print('Querying NASA Exoplanet Archive confirmed planets (pscomppars) ...')
confirmed_all = NasaExoplanetArchive.query_criteria(
    table='pscomppars',
    select='pl_name,hostname,discoverymethod,pl_orbper,pl_rade,st_teff,st_rad')
print(f'Total confirmed planets: {len(confirmed_all)}')

# ---- Filter: transit method only ----
is_transit = np.array(['Transit' in str(m) for m in confirmed_all['discoverymethod']])

# ---- Filter: period in TESS single-sector detectable range ----
orbital_periods = np.array([_safe_val(p) for p in confirmed_all['pl_orbper']])
period_finite   = np.isfinite(orbital_periods)
period_in_range = (orbital_periods >= MIN_PERIOD) & (orbital_periods <= MAX_PERIOD)

# Combine filters
filter_mask = is_transit & period_finite & period_in_range
filtered = confirmed_all[filter_mask]
print(f'Transit method + period {MIN_PERIOD}-{MAX_PERIOD}d: {len(filtered)} planets')

# ---- De-duplicate by host star (one planet per system) ----
all_hosts = [str(h).strip() for h in filtered['hostname']]
unique_hosts = list(dict.fromkeys(all_hosts))  # preserves order, removes duplicates
print(f'Unique host stars: {len(unique_hosts)}')

# ---- Random sample ----
rng = np.random.default_rng(RANDOM_SEED)
if len(unique_hosts) >= N_TARGETS:
    selected_hosts = rng.choice(np.array(unique_hosts), size=N_TARGETS, replace=False)
else:
    selected_hosts = np.array(unique_hosts)
    print(f'NOTE: only {len(unique_hosts)} eligible hosts found, using all')

# ---- Build target list with known planet info for reference ----
target_list = []
for host in selected_hosts:
    m = np.array([str(h).strip() == host for h in filtered['hostname']])
    rows = filtered[m]
    if len(rows) > 0:
        r = rows[0]
        target_list.append({
            'hostname':   host,
            'pl_name':    str(r['pl_name']),
            'pl_orbper':  _safe_val(r['pl_orbper']),
            'pl_rade':    _safe_val(r['pl_rade']),
            'st_teff':    _safe_val(r['st_teff']),
            'st_rad':     _safe_val(r['st_rad']),
        })

print(f'\nSelected {len(target_list)} targets. First 10 preview:')
for t in target_list[:10]:
    p_str = f"P={t['pl_orbper']:.2f}d" if np.isfinite(t['pl_orbper']) else 'P=?'
    r_str = f"Rp={t['pl_rade']:.1f}Rearth" if np.isfinite(t['pl_rade']) else ''
    print(f"  {t['hostname']:<25} {t['pl_name']:<25} {p_str:<10} {r_str}")

Querying NASA Exoplanet Archive confirmed planets (pscomppars) ...
Total confirmed planets: 6319
Transit method + period 0.5-40.0d: 4104 planets
Unique host stars: 3230

Selected 50 targets. First 10 preview:
  Kepler-980                Kepler-980 b              P=11.55d   Rp=2.7Rearth
  TrES-2                    TrES-2 b                  P=2.47d    Rp=15.2Rearth
  HAT-P-11                  HAT-P-11 b                P=4.89d    Rp=4.4Rearth
  Kepler-1505               Kepler-1505 b             P=30.86d   Rp=0.9Rearth
  TOI-406                   TOI-406.01                P=13.18d   Rp=2.1Rearth
  WASP-194                  WASP-194 b                P=3.18d    Rp=15.5Rearth
  HAT-P-62                  HAT-P-62 b                P=2.65d    Rp=12.0Rearth
  Kepler-837                Kepler-837 b              P=16.56d   Rp=2.6Rearth
  Kepler-447                Kepler-447 b              P=7.79d    Rp=18.5Rearth
  WASP-39                   WASP-39 b                 P=4.06d    Rp=14.3Rearth


## 3. Core Pipeline Functions

Same logic as Person2's `批量BLS_500目标.ipynb`, adapted to accept hostnames (not just TIC IDs).

In [11]:
def extract_arrays(lc_flat):
    '''Extract time, flux, flux_err from a flattened light curve.'''
    time = np.asarray(lc_flat.time.value, dtype=float)
    flux = np.asarray(lc_flat.flux.value, dtype=float)
    if getattr(lc_flat, 'flux_err', None) is not None:
        flux_err = np.asarray(lc_flat.flux_err.value, dtype=float)
    else:
        flux_err = None
    good = np.isfinite(time) & np.isfinite(flux)
    if flux_err is not None:
        good &= np.isfinite(flux_err)
        flux_err = flux_err[good]
    else:
        mad = np.nanmedian(np.abs(flux - np.nanmedian(flux)))
        flux_err = np.full(good.sum(), 1.4826 * mad)
    time = time[good]
    flux = flux[good]
    order = np.argsort(time)
    return time[order], flux[order], flux_err[order]


def run_bls(time, flux, flux_err, min_period=MIN_PERIOD, max_period=MAX_PERIOD,
            n_periods=15000, n_durations=10):
    '''Run BoxLeastSquares search; return (bls, result, best).'''
    bls = BoxLeastSquares(time, flux, dy=flux_err)
    periods = np.linspace(min_period, max_period, n_periods)
    durations = np.linspace(0.5/24.0, 6.0/24.0, n_durations)
    result = bls.power(periods, durations, objective='snr')
    idx = int(np.argmax(result.power))
    best = {
        'period':   float(result.period[idx]),
        'duration': float(result.duration[idx]),
        't0':       float(result.transit_time[idx]),
        'power':    float(result.power[idx]),
        'depth':    float(result.depth[idx]),
        'depth_err': float(result.depth_err[idx]),
        'depth_snr': float(result.depth_snr[idx]),
    }
    return bls, result, best


def bootstrap_errors(time, flux, flux_err, best, n_boot=N_BOOTSTRAP, rng_seed=42):
    '''Block bootstrap to estimate period / depth uncertainties.'''
    rng = np.random.default_rng(rng_seed)
    n = len(time)
    block_size = max(50, n // 50)
    n_blocks = int(np.ceil(n / block_size))
    lo = best['period'] * 0.95
    hi = best['period'] * 1.05
    boot_periods_grid = np.linspace(lo, hi, 300)
    durations = np.array([best['duration']])
    boot_periods, boot_depths = [], []
    for _ in range(n_boot):
        starts = rng.integers(0, n_blocks, size=n_blocks) * block_size
        idx = np.concatenate([np.arange(s, min(s + block_size, n)) for s in starts])
        idx = idx[idx < n]
        if len(idx) < 10:
            continue
        t_b, f_b, e_b = time[idx], flux[idx], flux_err[idx]
        order = np.argsort(t_b)
        t_b, f_b, e_b = t_b[order], f_b[order], e_b[order]
        try:
            bls_b = BoxLeastSquares(t_b, f_b, dy=e_b)
            res_b = bls_b.power(boot_periods_grid, durations, objective='snr')
            j = int(np.argmax(res_b.power))
            boot_periods.append(res_b.period[j])
            boot_depths.append(res_b.depth[j])
        except Exception:
            continue
    boot_periods = np.array(boot_periods)
    boot_depths  = np.array(boot_depths)
    return {
        'period_err': float(np.std(boot_periods)) if len(boot_periods) > 5 else np.nan,
        'depth_err':  float(np.std(boot_depths))  if len(boot_depths)  > 5 else np.nan,
    }


def vet_candidate(bls, best, snr_threshold=SNR_THRESHOLD):
    '''Apply false-positive rules to the candidate.'''
    flags = []
    if best['depth'] * 1e6 > 30000:
        flags.append('too_deep')
    if best['duration'] / best['period'] > 0.15:
        flags.append('too_long')
    if best['depth_snr'] < snr_threshold:
        flags.append('low_snr')
    return len(flags) > 0


def process_one_target(hostname, verbose=True):
    '''Download light curve by hostname, run BLS, return a 1-row summary DataFrame.'''
    if verbose:
        print(f'  Searching MAST for {hostname} ...')

    # Try multiple name variants for SPOC search
    search = None
    for name_variant in (hostname, hostname.replace(' ', '-'),
                          hostname.replace('-', ' ')):
        try:
            sr = lk.search_lightcurve(name_variant, author='SPOC')
            if sr is not None and len(sr) > 0:
                search = sr
                break
        except Exception:
            continue
    # Fallback: any TESS data
    if search is None:
        try:
            sr = lk.search_lightcurve(hostname)
            if sr is not None and len(sr) > 0:
                search = sr
        except Exception:
            pass
    if search is None or len(search) == 0:
        return None

    # Download first available sector
    try:
        lc = search[0].download(download_dir=TESS_CACHE_DIR)
    except Exception:
        return None
    if lc is None:
        return None

    sector = int(getattr(lc, 'sector', -1))
    if verbose:
        print(f'  Sector {sector}, flattening ...')
    lc_flat = lc.remove_nans().flatten(window_length=401)
    time, flux, flux_err = extract_arrays(lc_flat)
    if len(time) < 200:
        return None

    if verbose:
        print(f'  Running BLS ({len(time)} cadences) ...')
    bls, result, best = run_bls(time, flux, flux_err)
    if verbose:
        print(f'  P={best["period"]:.4f}d  depth={best["depth"]*1e6:.0f}ppm  SNR={best["depth_snr"]:.1f}')
    boot = bootstrap_errors(time, flux, flux_err, best)
    is_fp = vet_candidate(bls, best)
    summary = pd.DataFrame([{
        'Target':        hostname,
        'Sector':        sector,
        'Period (d)':    round(best['period'], 5),
        'Period Err (d)':round(boot['period_err'], 5) if np.isfinite(boot['period_err']) else np.nan,
        'Depth (ppm)':   round(best['depth'] * 1e6, 1),
        'Duration (hr)': round(best['duration'] * 24, 3),
        'SNR':           round(best['depth_snr'], 2),
        'False Positive':is_fp,
    }])
    return summary

## 4. Batch Runner

Process all 100 targets with timeout protection, intermediate saves, and failure logging (same as Person2 template).

In [12]:
import concurrent.futures as cf

all_tables = []
failed = []
t_start = timer.time()
completed_count = 0

# Check for existing checkpoint (resume support)
if os.path.exists(CHECKPOINT_CSV):
    existing = pd.read_csv(CHECKPOINT_CSV)
    all_tables = [existing]
    done_names = set(existing['Target'].tolist())
    print(f'Resumed from checkpoint: {len(existing)} targets already processed')
else:
    done_names = set()

for i, t in enumerate(target_list, 1):
    hostname = t['hostname']
    if hostname in done_names:
        continue

    p_label = f"P={t['pl_orbper']:.2f}d" if np.isfinite(t['pl_orbper']) else ''
    print(f'\n[{i}/{N_TARGETS}] {hostname}  ({t["pl_name"]} {p_label})')

    t0 = timer.time()
    with cf.ThreadPoolExecutor(max_workers=1) as ex:
        future = ex.submit(process_one_target, hostname, verbose=True)
        try:
            table = future.result(timeout=PER_TARGET_TIMEOUT)
            if table is not None:
                all_tables.append(table)
                completed_count += 1
                row = table.iloc[0]
                # Compare BLS-found period against NASA known period
                known_p = t['pl_orbper']
                found_p = row['Period (d)']
                if np.isfinite(known_p) and known_p > 0:
                    match = abs(found_p - known_p) / known_p < 0.05
                    match_str = 'MATCH' if match else f'diff: {abs(found_p-known_p)/known_p*100:.0f}%'
                else:
                    match_str = '?'
                print(f'  -> P_BLS={found_p:.4f}d (NASA={known_p:.4f}d, {match_str})  '
                      f'depth={row["Depth (ppm)"]:.0f}ppm  SNR={row["SNR"]:.1f}  FP={row["False Positive"]}')
            else:
                failed.append(hostname)
                print(f'  -> FAILED (no data / download error)')
        except cf.TimeoutError:
            failed.append(hostname)
            print(f'  -> TIMEOUT after {PER_TARGET_TIMEOUT}s')
        except Exception as e:
            failed.append(hostname)
            print(f'  -> FAILED: {type(e).__name__}: {str(e)[:80]}')

    print(f'  took {timer.time()-t0:.0f}s | completed: {completed_count} | failed: {len(failed)}')

    # Intermediate save every BATCH_SIZE
    if completed_count > 0 and completed_count % BATCH_SIZE == 0:
        checkpoint = pd.concat(all_tables, ignore_index=True)
        checkpoint.to_csv(CHECKPOINT_CSV, index=False, encoding='utf-8-sig')
        print(f'  [checkpoint saved: {len(checkpoint)} rows]')

print(f'\n{"="*60}')
print(f'BATCH COMPLETE in {timer.time()-t_start:.0f}s')
print(f'Succeeded: {completed_count}  |  Failed: {len(failed)}')
print(f'{"="*60}')


[1/50] Kepler-980  (Kepler-980 b P=11.55d)
  Searching MAST for Kepler-980 ...
  Sector 74, flattening ...
  Running BLS (18387 cadences) ...
  P=10.4863d  depth=10552ppm  SNR=5.0
  -> P_BLS=10.4863d (NASA=11.5510d, diff: 9%)  depth=10552ppm  SNR=5.0  FP=True
  took 24s | completed: 1 | failed: 0

[2/50] TrES-2  (TrES-2 b P=2.47d)
  Searching MAST for TrES-2 ...
  Sector 26, flattening ...
  Running BLS (16653 cadences) ...
  P=2.4699d  depth=13033ppm  SNR=109.0
  -> P_BLS=2.4699d (NASA=2.4706d, MATCH)  depth=13033ppm  SNR=109.0  FP=False
  took 25s | completed: 2 | failed: 0

[3/50] HAT-P-11  (HAT-P-11 b P=4.89d)
  Searching MAST for HAT-P-11 ...
  Sector 14, flattening ...
  Running BLS (18424 cadences) ...
  P=4.8874d  depth=2738ppm  SNR=95.1
  -> P_BLS=4.8874d (NASA=4.8880d, MATCH)  depth=2738ppm  SNR=95.1  FP=False
  took 31s | completed: 3 | failed: 0

[4/50] Kepler-1505  (Kepler-1505 b P=30.86d)
  Searching MAST for Kepler-1505 ...
  Sector 40, flattening ...
  Running BLS (196

  Sector 3, flattening ...
  Running BLS (13012 cadences) ...
  P=6.5860d  depth=1415ppm  SNR=6.4
  -> P_BLS=6.5860d (NASA=13.1757d, diff: 50%)  depth=1415ppm  SNR=6.4  FP=True
  took 18s | completed: 5 | failed: 0

[6/50] WASP-194  (WASP-194 b P=3.18d)
  Searching MAST for WASP-194 ...
  Sector 40, flattening ...
  Running BLS (19610 cadences) ...
  P=3.1835d  depth=6850ppm  SNR=45.3
  -> P_BLS=3.1835d (NASA=3.1834d, MATCH)  depth=6850ppm  SNR=45.2  FP=False
  took 30s | completed: 6 | failed: 0

[7/50] HAT-P-62  (HAT-P-62 b P=2.65d)
  Searching MAST for HAT-P-62 ...
  Sector 59, flattening ...
  Running BLS (16653 cadences) ...
  P=2.6463d  depth=7134ppm  SNR=51.0
  -> P_BLS=2.6463d (NASA=2.6453d, MATCH)  depth=7134ppm  SNR=51.0  FP=False
  took 20s | completed: 7 | failed: 0

[8/50] Kepler-837  (Kepler-837 b P=16.56d)
  Searching MAST for Kepler-837 ...
  Sector 80, flattening ...
  Running BLS (11444 cadences) ...
  P=5.8118d  depth=25541ppm  SNR=5.1
  -> P_BLS=5.8118d (NASA=16.560

  Sector 51, flattening ...
  Running BLS (8033 cadences) ...
  P=4.0552d  depth=20667ppm  SNR=117.7
  -> P_BLS=4.0552d (NASA=4.0553d, MATCH)  depth=20667ppm  SNR=117.7  FP=False
  took 16s | completed: 10 | failed: 0

[11/50] Kepler-1431  (Kepler-1431 b P=5.87d)
  Searching MAST for Kepler-1431 ...
  Sector 81, flattening ...
  Running BLS (15143 cadences) ...
  P=17.4071d  depth=31463ppm  SNR=4.8
  -> P_BLS=17.4071d (NASA=5.8660d, diff: 197%)  depth=31463ppm  SNR=4.8  FP=True
  took 30s | completed: 11 | failed: 0

[12/50] Kepler-910  (Kepler-910 b P=2.36d)
  Searching MAST for Kepler-910 ...
  Sector 54, flattening ...
  Running BLS (17898 cadences) ...
  P=6.3780d  depth=2205ppm  SNR=5.8
  -> P_BLS=6.3780d (NASA=2.3644d, diff: 170%)  depth=2205ppm  SNR=5.8  FP=True
  took 29s | completed: 12 | failed: 0

[13/50] Kepler-1121  (Kepler-1121 b P=13.15d)
  Searching MAST for Kepler-1121 ...
  Sector 41, flattening ...
  Running BLS (18322 cadences) ...
  P=5.7038d  depth=1086ppm  SNR=5.

  Sector 51, flattening ...
  Running BLS (8136 cadences) ...
  P=2.7253d  depth=4743ppm  SNR=550.1
  -> P_BLS=2.7253d (NASA=2.7240d, MATCH)  depth=4743ppm  SNR=550.1  FP=False
  took 18s | completed: 21 | failed: 0

[22/50] Kepler-632  (Kepler-632 b P=31.00d)
  Searching MAST for Kepler-632 ...
  Sector 74, flattening ...
  Running BLS (18604 cadences) ...
  P=6.3464d  depth=2776ppm  SNR=4.5
  -> P_BLS=6.3464d (NASA=30.9966d, diff: 80%)  depth=2776ppm  SNR=4.5  FP=True
  took 28s | completed: 22 | failed: 0

[23/50] Kepler-800  (Kepler-800 b P=14.13d)
  Searching MAST for Kepler-800 ...
  Sector 81, flattening ...
  Running BLS (16595 cadences) ...
  P=11.6424d  depth=33176ppm  SNR=4.7
  -> P_BLS=11.6424d (NASA=14.1318d, diff: 18%)  depth=33176ppm  SNR=4.7  FP=True
  took 26s | completed: 23 | failed: 0

[24/50] HAT-P-34  (HAT-P-34 b P=5.45d)
  Searching MAST for HAT-P-34 ...
  Sector 14, flattening ...
  Running BLS (13346 cadences) ...
  P=5.4536d  depth=3253ppm  SNR=60.8
  -> P_BLS

Could not resolve "TIC-77552382" to a sky position.


  Sector 8, flattening ...
  Running BLS (813 cadences) ...
  P=3.5075d  depth=3243ppm  SNR=8.8
  -> P_BLS=3.5075d (NASA=3.5103d, MATCH)  depth=3243ppm  SNR=8.8  FP=False
  took 22s | completed: 25 | failed: 0

[26/50] Kepler-1943  (Kepler-1943 b P=4.85d)
  Searching MAST for Kepler-1943 ...
  Sector 74, flattening ...
  Running BLS (18604 cadences) ...
  P=4.2501d  depth=6536ppm  SNR=5.1
  -> P_BLS=4.2501d (NASA=4.8502d, diff: 12%)  depth=6536ppm  SNR=5.1  FP=True
  took 31s | completed: 26 | failed: 0

[27/50] Kepler-765  (Kepler-765 b P=27.67d)
  Searching MAST for Kepler-765 ...
  Sector 75, flattening ...
  Running BLS (19474 cadences) ...
  P=24.2832d  depth=16146ppm  SNR=4.9
  -> P_BLS=24.2832d (NASA=27.6655d, diff: 12%)  depth=16146ppm  SNR=4.9  FP=True
  took 32s | completed: 27 | failed: 0

[28/50] Kepler-487  (Kepler-487 d P=2.42d)
  Searching MAST for Kepler-487 ...
  Sector 81, flattening ...
  Running BLS (18410 cadences) ...
  P=14.7341d  depth=34525ppm  SNR=5.3
  -> P_B

## 5. Save Final CSV and Summary

In [13]:
if all_tables:
    final = pd.concat(all_tables, ignore_index=True)
    # Ensure column order matches master_candidates.csv exactly
    COL_ORDER = ['Target', 'Sector', 'Period (d)', 'Period Err (d)',
                 'Depth (ppm)', 'Duration (hr)', 'SNR', 'False Positive']
    final = final[COL_ORDER]
    final.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
    final.to_csv(CHECKPOINT_CSV, index=False, encoding='utf-8-sig')

    print(f'Final table saved -> {OUTPUT_CSV}')
    print(f'Rows: {len(final)}  |  Columns: {list(final.columns)}')

    # Column consistency check against master_candidates.csv
    ref = pd.read_csv('inputs/01.csv', nrows=0)
    cols_match = list(final.columns) == list(ref.columns)
    print(f'Column match with master_candidates.csv: {cols_match}')
    if not cols_match:
        print('  Output:', list(final.columns))
        print('  Master:', list(ref.columns))

    # Statistics
    print(f'\nSummary statistics:')
    print(f'  SNR > 10, not FP : {len(final[(final["SNR"]>10) & (~final["False Positive"])])}')
    print(f'  SNR 7-10, not FP  : {len(final[(final["SNR"]>=7) & (final["SNR"]<=10) & (~final["False Positive"])])}')
    print(f'  Flagged FP        : {len(final[final["False Positive"]])}')
    print(f'\nFirst 10 rows:')
    print(final.head(10).to_string(index=False))
else:
    print('No targets succeeded. Check network / MAST access.')

if failed:
    print(f'\nFailed targets ({len(failed)}):')
    for f in failed[:20]:
        print(f'  {f}')
    if len(failed) > 20:
        print(f'  ... and {len(failed)-20} more')

Final table saved -> outputs\output.csv
Rows: 50  |  Columns: ['Target', 'Sector', 'Period (d)', 'Period Err (d)', 'Depth (ppm)', 'Duration (hr)', 'SNR', 'False Positive']
Column match with master_candidates.csv: True

Summary statistics:
  SNR > 10, not FP : 20
  SNR 7-10, not FP  : 1
  Flagged FP        : 29

First 10 rows:
     Target  Sector  Period (d)  Period Err (d)  Depth (ppm)  Duration (hr)    SNR  False Positive
 Kepler-980      74    10.48627         0.29412      10551.8           0.50   4.97            True
     TrES-2      26     2.46986         0.00048      13033.2           1.10 108.97           False
   HAT-P-11      14     4.88743         0.01171       2738.0           2.35  95.11           False
Kepler-1505      40    10.46520         0.26514       3335.3           0.50   4.88            True
    TOI-406       3     6.58604         0.11933       1414.8           1.70   6.43            True
   WASP-194      40     3.18355         0.00111       6850.3           1.70  4

## 6. Notes

- **Data source**: NASA Exoplanet Archive `pscomppars` table (all confirmed planets), filtered to transit-method discoveries with orbital periods 0.5-40 days (TESS single-sector detectable range).
- **BLS pipeline**: Identical to Person2's `bulkBLS_500targets.ipynb` — same `run_bls`, `bootstrap_errors`, `vet_candidate` functions.
- **Period verification**: The batch printout compares BLS-found period against NASA's known orbital period, showing `MATCH` when within 5%.
- **Output CSV**: Same 8 columns as `master_candidates.csv`, can be directly fed into the three-step quality control notebooks.